# Experimento: threshold normal vs fuzzy em 5 imagens de treino

Este notebook compara apenas as abordagens definidas em `THRESHOLD_APPROACHES`: threshold normal fixo e fuzzy por `alpha-cut`.

As saidas principais sao: parametros da execucao, threshold por imagem, status dos ostios, Dice por imagem e media de Dice por abordagem.


## 1. Ambiente

Configura o caminho do repositorio e importa as funcoes usadas no experimento.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, NOTEBOOK_CWD.parent, NOTEBOOK_CWD.parent.parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()


In [ ]:
import copy

import numpy as np
import pandas as pd
from skimage.morphology import ball

from utils.config_utils import load_config_json, scale_config_to_resolution
from utils.dataset_utils import get_data_splits
from utils.notebook_env import resolve_imagecas_base_path
from utils.processing import (
    binary_closing,
    binary_dilation,
    downscale_image_ndi,
    threshold_image_with_offset,
)
from utils.segmentation import (
    build_lcc_image_from_mask,
    detect_and_evaluate_ostia,
    fuzzy_trapezoid_threshold,
    get_or_compute_vesselness,
    get_or_detect_aorta_circles,
    get_or_segment_aorta,
    region_growing_segmentation,
)
from utils.utils import dice_score
from utils.utils.nifti_io import load_raw_img_and_label


## 2. Parametros do teste

Altere esta celula para escolher as imagens e quais abordagens de threshold devem rodar.


In [ ]:
CONFIG_PATH = REPO_ROOT / "config/pipeline_config.json"
BASE_PATH = resolve_imagecas_base_path()

TRAIN_SAMPLE_SIZE = 5
DOWNSCALE_FACTORS = (2, 2, 1)
THRESHOLD_APPROACHES = ["normal", "fuzzy_alpha_cut"]

MIN_HU = -300
FUZZY_MARGIN_HU = 160
FUZZY_ALPHA_CUT = 0.50

CONFIG = load_config_json(str(CONFIG_PATH), {})
CONFIG["DOWNSCALE_FACTORS"] = list(DOWNSCALE_FACTORS)
CONFIG["LOAD_CACHE"] = False
CONFIG["SAVE_CACHE"] = False
RUN_CONFIG = scale_config_to_resolution(copy.deepcopy(CONFIG))

MAX_THRESHOLD_PERCENTILE = CONFIG.get("MAX_THRESHOLD_PERCENTILE", 99.7)
train_ids, val_ids, test_ids, all_ids = get_data_splits(str(BASE_PATH))
SAMPLE_IMAGE_IDS = train_ids[:TRAIN_SAMPLE_SIZE]

config_df = pd.DataFrame(
    [
        {
            "train_sample_size": TRAIN_SAMPLE_SIZE,
            "sample_image_ids": SAMPLE_IMAGE_IDS,
            "threshold_approaches": THRESHOLD_APPROACHES,
            "base_path": str(BASE_PATH),
            "downscale_factors": DOWNSCALE_FACTORS,
            "min_hu": MIN_HU,
            "normal_max_threshold_percentile": MAX_THRESHOLD_PERCENTILE,
            "fuzzy_margin_hu": FUZZY_MARGIN_HU,
            "fuzzy_alpha_cut": FUZZY_ALPHA_CUT,
            "load_cache": RUN_CONFIG["LOAD_CACHE"],
            "save_cache": RUN_CONFIG["SAVE_CACHE"],
        }
    ]
)
config_df


## 3. Funcoes auxiliares

As funcoes abaixo executam as abordagens definidas em `THRESHOLD_APPROACHES`, detectam ostios e calculam o Dice da segmentacao arterial padrao.


In [ ]:
def load_sample_image(img_id: int) -> dict:
    """Carrega a imagem, a label e os metadados reduzidos de um caso."""
    img_path = BASE_PATH / f"{img_id}.img.nii.gz"
    label_path = BASE_PATH / f"{img_id}.label.nii.gz"
    nii_img, nii_label = load_raw_img_and_label(str(img_path), str(label_path))
    image = nii_img.get_fdata(dtype=np.float32)
    label = nii_label.get_fdata(dtype=np.float32).astype(np.uint8)
    spacing = tuple(float(value) for value in nii_img.header.get_zooms()[:3])
    down_label = downscale_image_ndi(label, DOWNSCALE_FACTORS, order=0).astype(np.uint8)
    scaled_spacing = tuple(
        spacing[idx] * DOWNSCALE_FACTORS[idx] for idx in range(len(DOWNSCALE_FACTORS))
    )
    return {
        "img_id": img_id,
        "image": image,
        "down_label": down_label,
        "spacing": spacing,
        "scaled_spacing": scaled_spacing,
        "image_shape": image.shape,
        "label_shape": label.shape,
        "down_label_shape": down_label.shape,
    }


def build_threshold_inputs(img_id: int, image: np.ndarray) -> tuple[dict, pd.DataFrame]:
    """Gera as imagens de entrada para cada abordagem ativa."""
    down_image = downscale_image_ndi(image, DOWNSCALE_FACTORS, order=3).astype(np.float32)
    max_hu = float(np.percentile(down_image, MAX_THRESHOLD_PERCENTILE))
    approach_inputs = {}
    records = []

    if "normal" in THRESHOLD_APPROACHES:
        _, normal_mask, _ = threshold_image_with_offset(
            down_image,
            min_val=MIN_HU,
            max_val=int(max_hu),
        )
        normal_lcc_image, normal_lcc_mask = build_lcc_image_from_mask(
            down_image,
            normal_mask,
            offset=abs(MIN_HU),
            per_slice=True,
        )
        approach_inputs["normal"] = normal_lcc_image
        records.append(
            {
                "img_id": img_id,
                "approach": "normal",
                "threshold_mode": "crisp",
                "min_hu": float(MIN_HU),
                "max_hu": max_hu,
                "threshold_voxels": int(normal_mask.sum()),
                "lcc_voxels": int(normal_lcc_mask.sum()),
            }
        )

    if "fuzzy_alpha_cut" in THRESHOLD_APPROACHES:
        fuzzy_membership = fuzzy_trapezoid_threshold(
            down_image,
            min_hu=MIN_HU,
            max_hu=max_hu,
            margin_hu=FUZZY_MARGIN_HU,
        )
        fuzzy_mask = fuzzy_membership >= FUZZY_ALPHA_CUT
        fuzzy_lcc_image, fuzzy_lcc_mask = build_lcc_image_from_mask(
            down_image,
            fuzzy_mask,
            offset=abs(MIN_HU),
            per_slice=True,
        )
        approach_inputs["fuzzy_alpha_cut"] = fuzzy_lcc_image
        records.append(
            {
                "img_id": img_id,
                "approach": "fuzzy_alpha_cut",
                "threshold_mode": "fuzzy_alpha_cut",
                "min_hu": float(MIN_HU),
                "max_hu": max_hu,
                "threshold_voxels": int(fuzzy_mask.sum()),
                "lcc_voxels": int(fuzzy_lcc_mask.sum()),
            }
        )

    return approach_inputs, pd.DataFrame(records)


def run_detection_stage(
    img_id: int,
    approach: str,
    lcc_image: np.ndarray,
    label: np.ndarray,
    scaled_spacing: tuple[float, float, float],
    config: dict,
) -> dict:
    """Executa aorta + ostios sem criar cache."""
    stage_root = REPO_ROOT / "output" / "tmp_fuzzy_threshold_only_no_cache" / str(img_id)
    result = {
        "img_id": img_id,
        "approach": approach,
        "ostia_found": False,
        "ostia_status": "not_evaluated",
        "both_correct": False,
        "both_tolerable": False,
        "left_dist_mm": np.inf,
        "right_dist_mm": np.inf,
        "ostia_left": None,
        "ostia_right": None,
        "num_circles": 0,
        "aorta_voxels": 0,
        "ostia_error": None,
    }
    vesselness_ostios = get_or_compute_vesselness(
        str(img_id),
        lcc_image,
        cache_dir=str(stage_root / approach / "vesselness_ostios_cache"),
        vesselness_config=config["VESSELNESS_AORTA"],
        load_cache=config["LOAD_CACHE"],
        save_cache=config["SAVE_CACHE"],
        use_gpu=config.get("USE_GPU", False),
    )
    detected_circles = get_or_detect_aorta_circles(
        str(img_id),
        lcc_image,
        DOWNSCALE_FACTORS,
        scaled_spacing,
        config["CIRCLE_DETECTION"],
        stage_root / approach,
        load_cache=config["LOAD_CACHE"],
        save_cache=config["SAVE_CACHE"],
    )
    result["num_circles"] = len(detected_circles)
    aorta_mask = get_or_segment_aorta(
        str(img_id),
        lcc_image,
        detected_circles,
        config["LEVEL_SET"],
        stage_root / approach,
        load_cache=config["LOAD_CACHE"],
        save_cache=config["SAVE_CACHE"],
        use_gpu=config.get("USE_GPU", False),
    )
    result["aorta_voxels"] = int(aorta_mask.sum())
    try:
        ostia_eval = detect_and_evaluate_ostia(
            aorta_mask,
            vesselness_ostios,
            label,
            scaled_spacing,
            config,
        )
    except ValueError as exc:
        result["ostia_status"] = "not_found"
        result["ostia_error"] = str(exc)
        return result

    missing_ostia = [
        side
        for side, ostium in (
            ("left", ostia_eval.get("ostia_left")),
            ("right", ostia_eval.get("ostia_right")),
        )
        if ostium is None
    ]
    if missing_ostia:
        result["ostia_status"] = "not_found"
        result["ostia_error"] = "Ostio(s) nao encontrado(s): " + ", ".join(missing_ostia)
        return result

    if ostia_eval["both_correct"]:
        ostia_status = "both_correct"
    elif ostia_eval["both_tolerable"]:
        ostia_status = "both_tolerable"
    else:
        ostia_status = "found_but_wrong"

    result.update(
        {
            "ostia_found": True,
            "ostia_status": ostia_status,
            "both_correct": bool(ostia_eval["both_correct"]),
            "both_tolerable": bool(ostia_eval["both_tolerable"]),
            "left_dist_mm": ostia_eval["left_info"]["physical_dist"],
            "right_dist_mm": ostia_eval["right_info"]["physical_dist"],
            "ostia_left": tuple(map(int, ostia_eval["ostia_left"])),
            "ostia_right": tuple(map(int, ostia_eval["ostia_right"])),
            "label_artery": ostia_eval["label_artery"],
        }
    )
    return result


def postprocess_artery_mask(mask: np.ndarray, config: dict) -> np.ndarray:
    """Aplica apenas o pos-processamento morfologico usado no pipeline."""
    post_config = config["POSTPROCESSING"]
    closed_mask = binary_closing(
        mask > 0,
        structure=ball(post_config["closing_radius"]),
        gpu=config.get("USE_GPU", False),
    )
    return binary_dilation(
        closed_mask,
        structure=ball(post_config["dilation_radius"]),
        gpu=config.get("USE_GPU", False),
    )


def build_region_growing_params(vesselness_artery: np.ndarray, config: dict) -> dict:
    rg_config = config["REGION_GROWING"]
    return {
        "threshold": (vesselness_artery.max() - vesselness_artery.min())
        / rg_config["threshold_divisor"],
        "max_volume": rg_config["max_volume"],
        "min_vesselness": vesselness_artery.max()
        * rg_config["min_vesselness_fraction"],
        "relaxed_floor_factor": rg_config["relaxed_floor_factor"],
        "switch_at_voxels": rg_config["switch_at_voxels"],
        "comparison_window": rg_config["comparison_window"],
        "smooth_relaxation": rg_config["smooth_relaxation"],
        "verbose": False,
    }


def standard_region_growing_from_seeds(
    vesselness_artery: np.ndarray,
    seeds,
    config: dict,
) -> np.ndarray:
    params = build_region_growing_params(vesselness_artery, config)
    combined = np.zeros_like(vesselness_artery, dtype=np.uint8)
    for seed in seeds:
        if seed is not None:
            combined |= region_growing_segmentation(
                vesselness_artery,
                seed_point=seed,
                **params,
            )
    return combined.astype(np.uint8)


def run_segmentation_stage(
    img_id: int,
    detection_result: dict,
    lcc_image: np.ndarray,
    config: dict,
) -> dict:
    """Executa region growing padrao com e sem pos-processamento."""
    result = {
        "img_id": img_id,
        "approach": detection_result["approach"],
        "segmentation_attempted": False,
        "proceeded_with_bad_ostia": False,
        "dice_artery": np.nan,
        "artery_voxels": 0,
        "segmentation_error": detection_result["ostia_error"],
        "case_rows": [],
    }
    if not detection_result["ostia_found"]:
        return result

    result["segmentation_attempted"] = True
    result["proceeded_with_bad_ostia"] = not (
        detection_result["both_correct"] or detection_result["both_tolerable"]
    )
    stage_root = REPO_ROOT / "output" / "tmp_fuzzy_threshold_only_no_cache" / str(img_id)
    vesselness_artery = get_or_compute_vesselness(
        str(img_id),
        lcc_image,
        cache_dir=str(stage_root / detection_result["approach"] / "vesselness_artery_cache"),
        vesselness_config=config["VESSELNESS_ARTERY"],
        load_cache=config["LOAD_CACHE"],
        save_cache=config["SAVE_CACHE"],
        use_gpu=config.get("USE_GPU", False),
    )
    raw_mask = standard_region_growing_from_seeds(
        vesselness_artery,
        [detection_result["ostia_left"], detection_result["ostia_right"]],
        config,
    )
    masks = {
        "without_morphology": raw_mask,
        "with_morphology": postprocess_artery_mask(raw_mask, config),
    }
    for suffix, mask in masks.items():
        result["case_rows"].append(
            {
                "img_id": img_id,
                "approach": detection_result["approach"],
                "segmentation_case": f"standard_region_growing_pipeline_{suffix}",
                "uses_morphology": suffix == "with_morphology",
                "dice_artery": float(dice_score(mask, detection_result["label_artery"])),
                "artery_voxels": int(mask.sum()),
                "segmentation_attempted": result["segmentation_attempted"],
                "proceeded_with_bad_ostia": result["proceeded_with_bad_ostia"],
                "segmentation_error": result["segmentation_error"],
            }
        )

    final_row = next(row for row in result["case_rows"] if row["uses_morphology"])
    result["dice_artery"] = final_row["dice_artery"]
    result["artery_voxels"] = final_row["artery_voxels"]
    return result


def run_sample_experiment(img_id: int) -> dict:
    sample = load_sample_image(img_id)
    approach_inputs, threshold_df = build_threshold_inputs(img_id, sample["image"])
    detection_results = {
        approach: run_detection_stage(
            img_id,
            approach,
            lcc_image,
            sample["down_label"],
            sample["scaled_spacing"],
            RUN_CONFIG,
        )
        for approach, lcc_image in approach_inputs.items()
    }
    ostia_df = pd.DataFrame(
        [
            {
                "img_id": result["img_id"],
                "approach": result["approach"],
                "ostia_status": result["ostia_status"],
                "ostia_found": result["ostia_found"],
                "both_correct": result["both_correct"],
                "both_tolerable": result["both_tolerable"],
                "left_dist_mm": result["left_dist_mm"],
                "right_dist_mm": result["right_dist_mm"],
                "ostia_left": result["ostia_left"],
                "ostia_right": result["ostia_right"],
                "num_circles": result["num_circles"],
                "aorta_voxels": result["aorta_voxels"],
                "ostia_error": result["ostia_error"],
            }
            for result in detection_results.values()
        ]
    )
    segmentation_results = {
        approach: run_segmentation_stage(
            img_id,
            detection_result,
            approach_inputs[approach],
            RUN_CONFIG,
        )
        for approach, detection_result in detection_results.items()
    }
    segmentation_df = pd.DataFrame(
        [
            {key: value for key, value in result.items() if key != "case_rows"}
            for result in segmentation_results.values()
        ]
    )
    segmentation_comparison_df = pd.DataFrame(
        row for result in segmentation_results.values() for row in result["case_rows"]
    )
    sample_info_df = pd.DataFrame(
        [
            {
                "img_id": img_id,
                "image_shape": sample["image_shape"],
                "label_shape": sample["label_shape"],
                "down_label_shape": sample["down_label_shape"],
                "spacing_mm": sample["spacing"],
                "scaled_spacing_mm": sample["scaled_spacing"],
            }
        ]
    )
    return {
        "sample_info_df": sample_info_df,
        "threshold_df": threshold_df,
        "ostia_df": ostia_df,
        "segmentation_df": segmentation_df,
        "segmentation_comparison_df": segmentation_comparison_df,
    }


def concat_outputs(outputs: list[dict], key: str) -> pd.DataFrame:
    frames = [output[key] for output in outputs if not output[key].empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


## 4. Amostras do treino

Os IDs abaixo vêm de `config/imagecas_splits.json` via `get_data_splits`, ou seja, usam a mesma divisao fixa do pipeline.


In [ ]:
sample_ids_df = pd.DataFrame(
    {
        "sample_order": range(1, len(SAMPLE_IMAGE_IDS) + 1),
        "img_id": SAMPLE_IMAGE_IDS,
        "split": "train",
    }
)
sample_ids_df


## 5. Execucao das abordagens

Esta celula roda as tres abordagens de threshold para cada uma das 5 imagens de treino e, em seguida, usa a segmentacao arterial padrao do pipeline.


In [ ]:
sample_outputs = [run_sample_experiment(img_id) for img_id in SAMPLE_IMAGE_IDS]

sample_info_df = concat_outputs(sample_outputs, "sample_info_df")
threshold_df = concat_outputs(sample_outputs, "threshold_df")
ostia_df = concat_outputs(sample_outputs, "ostia_df")
segmentation_df = concat_outputs(sample_outputs, "segmentation_df")
segmentation_comparison_df = concat_outputs(sample_outputs, "segmentation_comparison_df")

threshold_meta_cols = [
    "img_id",
    "approach",
    "threshold_mode",
    "min_hu",
    "max_hu",
    "threshold_voxels",
    "lcc_voxels",
]

if not segmentation_comparison_df.empty:
    segmentation_comparison_df = threshold_df[threshold_meta_cols].merge(
        segmentation_comparison_df,
        on=["img_id", "approach"],
        how="right",
    )
    segmentation_comparison_df = segmentation_comparison_df.sort_values(
        ["img_id", "approach", "uses_morphology"],
        na_position="first",
    ).reset_index(drop=True)

ostia_df = threshold_df[
    ["img_id", "approach", "threshold_mode", "min_hu", "max_hu"]
].merge(ostia_df, on=["img_id", "approach"], how="right")

threshold_df.sort_values(["img_id", "approach"])[
    [
        "img_id",
        "approach",
        "threshold_mode",
        "min_hu",
        "max_hu",
        "threshold_voxels",
        "lcc_voxels",
    ]
]


## 6. Deteccao da aorta e dos ostios

Resumo por imagem e abordagem: circulos detectados, tamanho da mascara da aorta, status dos ostios e distancias ate a label.


In [ ]:
ostia_df.sort_values(["img_id", "approach"])[
    [
        "img_id",
        "approach",
        "ostia_status",
        "both_correct",
        "both_tolerable",
        "left_dist_mm",
        "right_dist_mm",
        "num_circles",
        "aorta_voxels",
        "ostia_error",
    ]
]


## 7. Dice score por imagem

A tabela principal usa o resultado com pos-processamento morfologico, equivalente ao Dice final do pipeline. A tabela longa `segmentation_comparison_df` continua disponivel caso voce queira comparar sem morfologia.


In [ ]:
morphology_dice_df = segmentation_comparison_df[
    segmentation_comparison_df["uses_morphology"]
].copy()

dice_by_image_df = (
    morphology_dice_df.pivot_table(
        index="img_id",
        columns="approach",
        values="dice_artery",
        aggfunc="first",
    )
    .reset_index()
    .sort_values("img_id")
)
dice_by_image_df.columns.name = None
dice_by_image_df


## 8. Media de Dice por abordagem

Resumo calculado sobre as 5 imagens de treino, usando o resultado com pos-processamento morfologico.


In [ ]:
dice_summary_df = (
    morphology_dice_df.groupby(["approach", "threshold_mode"], as_index=False)
    .agg(
        mean_dice=("dice_artery", "mean"),
        std_dice=("dice_artery", "std"),
        min_dice=("dice_artery", "min"),
        max_dice=("dice_artery", "max"),
        evaluated_images=("img_id", "nunique"),
        mean_threshold_hu=("max_hu", "mean"),
        mean_threshold_voxels=("threshold_voxels", "mean"),
        mean_lcc_voxels=("lcc_voxels", "mean"),
    )
    .sort_values("mean_dice", ascending=False)
    .reset_index(drop=True)
)
dice_summary_df
